# ComfyUI on Colab — AI image/video generation
Node-based Stable Diffusion studio on your free Colab GPU.

**Phone steps:** Runtime → Run all → open the tunnel URL from the last cell.
Build workflows visually: text-to-image, img2img, upscaling, animation frames.

In [ ]:
# 1) system + ComfyUI
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install -q -r requirements.txt
!pip install -q onnxruntime-gpu  # optional, for some nodes

In [ ]:
# 2) a fast light model: SDXL Turbo (~7GB, great on free T4)
!mkdir -p models/checkpoints
!wget -q --show-progress -O models/checkpoints/sd_xl_turbo_1.0_fp16.safetensors https://huggingface.co/stabilityai/sdxl-turbo/resolve/main/sd_xl_turbo_1.0_fp16.safetensors

In [ ]:
# 3) launch + public link (cloudflared works on Colab's network)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cf
!chmod +x /tmp/cf
import subprocess, time
srv = subprocess.Popen(['python', 'main.py', '--port', '8188'])
time.sleep(8)
tun = subprocess.Popen(['/tmp/cf', 'tunnel', '--url', 'http://localhost:8188'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
import re
url = ''
for _ in range(60):
    line = tun.stdout.readline()
    m = re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('OPEN ON YOUR PHONE:', url)
srv.wait()  # keep alive; Runtime -> Interrupt to stop

## Tips
- Queue a prompt, then **Queue Prompt** — 4 steps is enough with Turbo.
- Save workflows you like (the .json) — they go straight into the repo.
- For video: add AnimateDiff nodes (install via ComfyUI Manager in the UI).
- Disk: Colab gives ~100GB; models live in `models/checkpoints/`.